# filtrage des datasets de datagouv

In [1]:
import pandas as pd
import re


In [3]:
# lecture des datasets exportés en parquet depuis l'api datagouv dans lecture_datasets_api.ipynb
dsdatagouv = pd.read_parquet("dsdatagouv.parquet")

In [4]:
dsdatagouv.shape

(73983, 13)

## recherche basique OU sur des mots-clés 

In [5]:
keywords = [
    "transport", "mobilité", "route", "passager", "fret", "parking", "stationnement", "voirie", "accessibilité", "enquête", "horaire", "calendrier", 
    "vélo", "trafic", "piéton", "voiture", "véhicule", "carburant", "accident", "sécurité routière", "train", "transport public", "avion", "fluvial", "maritime",
    "covoiturage", "autopartage", "bruit", "qualité de l'air"
]


In [6]:
## enlever les accents et passer tout en minuscules pour faciliter la recherche
"mobilité".translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))

'mobilite'

In [7]:
# Construire un pattern regex : mot1|mot2|mot3...
pattern = "|".join(re.escape(kw.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))) for kw in keywords)

# recherche dans les champs titre et description
# Concaténer title + description pour chercher dans les deux
#text = (
#    dsdatagouv["title"].apply(lambda x: x.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))).fillna("") + " " + dsdatagouv["description"].apply(lambda x: x.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))).fillna("")
#)
text = (
    dsdatagouv["tags"].apply(lambda x: " ".join(x)).apply(lambda x: x.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))).fillna("") 
)
    

In [9]:
pattern

"transport|mobilite|route|passager|fret|parking|stationnement|voirie|accessibilite|enquete|horaire|calendrier|velo|trafic|pieton|voiture|vehicule|carburant|accident|securite\\ routiere|train|transport\\ public|avion|fluvial|maritime|covoiturage|autopartage|bruit|qualite\\ de\\ l'air"

### recherche sur les tags

In [15]:
mask = text.str.contains(pattern, case=False, na=False, regex=True)
dsdatagouv_mob_tags = dsdatagouv[mask].copy()

In [16]:
dsdatagouv_mob_tags.shape
# environ 8000 datasets pour 75000 au départ

(8209, 13)

### recherche dans les champs titre et description

In [18]:
# Concaténer title + description pour chercher dans les deux
text = (
    dsdatagouv["title"].apply(lambda x: x.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))).fillna("") + " " + dsdatagouv["description"].apply(lambda x: x.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))).fillna("")
)
mask = text.str.contains(pattern, case=False, na=False, regex=True)
dsdatagouv_mob_titledesc = dsdatagouv[mask].copy()

In [20]:
# la recherche des chaines prend 1' ; env 19k datasets trouvés pour 75k au départ
dsdatagouv_mob_titledesc.shape

(19740, 13)

In [33]:
dsdatagouv_mob_tags.to_parquet("dsdatagouv_mob.parquet")

In [34]:
# on enregistre la sélection par tags

### comparaison entre requête par tag et par titre+descr

In [36]:
dsmob=pd.merge(dsdatagouv_mob_tags, dsdatagouv_mob_titledesc,how="outer",on="id")

In [38]:
dsmob.shape

(21875, 25)

In [39]:
dsmob.columns

Index(['id', 'title_x', 'description_x', 'org_x', 'org_id_x', 'tags_x',
       'views_x', 'ressources_nb_x', 'ressources_texte_x', 'downloads_x',
       'license_x', 'frequency_x', 'last_modified_x', 'title_y',
       'description_y', 'org_y', 'org_id_y', 'tags_y', 'views_y',
       'ressources_nb_y', 'ressources_texte_y', 'downloads_y', 'license_y',
       'frequency_y', 'last_modified_y'],
      dtype='object')

21875 au total en fusionnant les datasets cherchés par tags et ceux cherchés par titre+desc

In [89]:
xx=dsmob['title_x'].isna().apply(lambda x: str(x))+ dsmob['title_y'].isna().apply(lambda x: str(x))

In [95]:
xx.value_counts()

TrueFalse     13666
FalseFalse     6074
FalseTrue      2135
Name: count, dtype: int64

sur les 8209 datasets trouvés par les tags, 6074 sont dans la liste des datasets trouvés à partir de titre + description et 2153 n'y sont pas 
sur les 19740 datasets trouvés sur titre+desc, 6074 sont dans la liste des datasets trouvés à partir des tags et 13666 n'y sont pas 


In [99]:
dsmob[xx=="FalseTrue"][['title_x','description_x','tags_x']]

,title_x,description_x,tags_x
23,Actualités du STAR,,"[actualites, transports]"
25,Agenda 21,Plaquette de présentation de l'Agenda 21 dépar...,"[agenda-21, democratie, developpement-durable,..."
27,Agriculture biologique 2008-2011 - productions...,Surfaces en conversion et bio par département ...,"[agriculture, agriculture-biologique, developp..."
36,Aires de livraison,Localisation des aires de livraison,"[deplacement, livraison, mobilib, mobilite, st..."
71,CAD 5: Engagement (ou versements bruts) bilaté...,CAD 5: Engagement (ou versements bruts) bilaté...,"[aide-au-developpement, engagements, versement..."
...,...,...,...
21855,[géolittoral] 2026 Indice de sensibilité morph...,https://www.geocatalogue.fr/geonetwork/srv/fre...,"[inspire, occitanie, polmar, regions-maritimes]"
21856,[géolittoral] 2026 Indice de sensibilité envir...,https://www.geocatalogue.fr/geonetwork/srv/fre...,"[inspire, occitanie, polmar, regions-maritimes]"
21861,DCSMM - Sous-régions marines (France),Parties françaises des sous-régions marines eu...,"[actions-concretes, boulogne-sur-mer, brest, d..."
21867,test new version de gn,probleme de modification de metadonnee,"[cc-presquile-de-crozon-aulne-maritime-epci, p..."


In [100]:
dsmob[xx=="FalseTrue"][['tags_x']]

,tags_x
23,"[actualites, transports]"
25,"[agenda-21, democratie, developpement-durable,..."
27,"[agriculture, agriculture-biologique, developp..."
36,"[deplacement, livraison, mobilib, mobilite, st..."
71,"[aide-au-developpement, engagements, versement..."
...,...
21855,"[inspire, occitanie, polmar, regions-maritimes]"
21856,"[inspire, occitanie, polmar, regions-maritimes]"
21861,"[actions-concretes, boulogne-sur-mer, brest, d..."
21867,"[cc-presquile-de-crozon-aulne-maritime-epci, p..."


In [101]:
dsmob.to_csv('dsmob.csv')